In [1]:
# ==========================================
# 📦 CELL 1 — INSTALL DEPENDENCIES
# ==========================================
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.51.0' 'diffusers>=0.32.0' 'accelerate>=0.30.0' 'safetensors'
!pip install -q fastapi uvicorn nest-asyncio python-multipart
!npm install -g localtunnel &> /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb &> /dev/null
print("✅ Cloudflare Tunnel is ready.")

print('✅ All dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.6 MB/s eta 0:00:00
✅ Cloudflare Tunnel is ready.
✅ All dependencies installed.


In [2]:
# ==========================================
# ⚡ CELL 2 — PRE-LOAD AI ENGINE (MAX PERFORMANCE)
# ==========================================
import os, json, torch, gc
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler

# Clear VRAM junk
gc.collect()
torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hardware level optimizations
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

IMG_MODEL_ID = 'John6666/janku-v5-nsfw-trained-noobai-rou-wei-illustrious-xl-v50-sdxl'
LORA_MODEL_ID = 'ByteDance/SDXL-Lightning'

print(f'⏳ Loading Engine into VRAM...')
loaded_model = StableDiffusionXLPipeline.from_pretrained(
    IMG_MODEL_ID, torch_dtype=torch.float16, use_safetensors=True
).to(DEVICE)

# Channels Last = massive speedup for T4 GPUs
loaded_model.unet.to(memory_format=torch.channels_last)

loaded_model.scheduler = EulerDiscreteScheduler.from_config(
    loaded_model.scheduler.config, timestep_spacing="trailing"
)

print(f'⏳ Injecting Lightning LoRA...')
loaded_model.load_lora_weights(
    LORA_MODEL_ID,
    weight_name="sdxl_lightning_4step_lora.safetensors",
    adapter_name="lightning_base"
)
loaded_model.set_adapters(["lightning_base"], adapter_weights=[1.0])

# --- THE SPEED REVEAL (Corrected Syntax) ---
loaded_model.disable_attention_slicing()
loaded_model.vae.disable_tiling()
loaded_model.vae.disable_slicing()

# --- CUSTOM CIVITAI LORA RUNTIME REGISTRY ---
CUSTOM_LORA_DIR = '/content/custom_noobai_loras'
os.makedirs(CUSTOM_LORA_DIR, exist_ok=True)
CUSTOM_LORA_STATE_PATH = os.path.join(CUSTOM_LORA_DIR, 'lora_registry.json')
CUSTOM_LORA_REGISTRY = []
CUSTOM_LORA_BY_KEY = {}

if os.path.exists(CUSTOM_LORA_STATE_PATH):
    try:
        with open(CUSTOM_LORA_STATE_PATH, 'r', encoding='utf-8') as f:
            saved = json.load(f)
        if isinstance(saved, list):
            CUSTOM_LORA_REGISTRY = saved
    except Exception as e:
        print(f'⚠️ Could not restore saved LoRA registry: {e}')

def save_custom_lora_registry():
    with open(CUSTOM_LORA_STATE_PATH, 'w', encoding='utf-8') as f:
        json.dump(CUSTOM_LORA_REGISTRY, f, indent=2)

for item in CUSTOM_LORA_REGISTRY:
    try:
        if item.get('path') and os.path.exists(item['path']):
            loaded_model.load_lora_weights(item['path'], adapter_name=item['adapter_name'])
            item['loaded'] = True
        else:
            item['loaded'] = False
    except Exception as e:
        item['loaded'] = False
        print(f"⚠️ Could not restore {item.get('name', 'custom LoRA')}: {e}")

save_custom_lora_registry()

print('✅ Engine UNLEASHED. Ready for 1-second bakes.')
print(f'📁 Custom LoRA folder: {CUSTOM_LORA_DIR}')


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


⏳ Loading Engine into VRAM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

⏳ Injecting Lightning LoRA...


sdxl_lightning_4step_lora.safetensors:   0%|          | 0.00/394M [00:00<?, ?B/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


✅ Engine UNLEASHED. Ready for 1-second bakes.


In [ ]:
# ==========================================
# 🚀 CELL 3 — LAUNCH STUDIO & TUNNEL (The Definitive Edition)
# ==========================================
# 1. Kill any ghost processes blocking the port
!fuser -k 8000/tcp

import nest_asyncio
import uvicorn
from fastapi import FastAPI, Query, Body
from fastapi.responses import StreamingResponse, HTMLResponse, JSONResponse
import io, os, json, time, torch, random, threading, urllib.request, urllib.parse, subprocess, asyncio, re
import __main__

# Apply nest_asyncio to prevent loop conflicts
nest_asyncio.apply()

app = FastAPI()
gpu_lock = threading.Lock()

# --- SAFER DEVICE DETECTION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- THE FRONTEND ---
html_content = r"""
<!DOCTYPE html>
<html>
<head>
    <title>Vision Studio Feed</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
    <style>
        :root {
            --accent: #e11d48;
            --accent-dark: #be123c;
            --bg: #000;
            --panel-bg: rgba(12,12,12,0.96);
            --border: #2a2a2a;
            --text: #ddd;
            --muted: #888;
        }

        * { box-sizing: border-box; }
        body, html { margin: 0; padding: 0; height: 100%; background: var(--bg); color: var(--text); font-family: 'Segoe UI', sans-serif; overflow: hidden; }

        /* ===== FEED ===== */
        #feed {
            height: 100vh; width: 100vw;
            overflow-y: scroll;
            scroll-snap-type: y mandatory;
            -webkit-overflow-scrolling: touch;
            scrollbar-width: none;
        }
        #feed::-webkit-scrollbar { display: none; }

        .post {
            height: 100vh; width: 100vw;
            scroll-snap-align: start;
            display: flex; justify-content: center; align-items: center;
            position: relative;
            background: #050505;
            overflow: hidden;
            transition: background 1.2s ease;
        }

        /* Ken Burns wrapper */
        .post-img-wrap { position: absolute; inset: 0; overflow: hidden; }
        .post img {
            height: 100%; width: 100%;
            object-fit: contain;
            animation: kenBurns 12s ease-in-out infinite alternate;
            transform-origin: center center;
            transition: opacity 0.6s ease;
        }
        @keyframes kenBurns {
            0%   { transform: scale(1.0) translate(0px, 0px); }
            33%  { transform: scale(1.04) translate(-8px, -5px); }
            66%  { transform: scale(1.02) translate(6px, 4px); }
            100% { transform: scale(1.05) translate(-4px, 6px); }
        }

        /* Blur-up placeholder */
        .blur-placeholder {
            position: absolute; inset: 0;
            display: flex; flex-direction: column;
            align-items: center; justify-content: center;
            gap: 16px; background: #0a0a0a; z-index: 2;
        }
        .pulse-ring {
            width: 56px; height: 56px; border-radius: 50%;
            border: 2px solid var(--accent);
            animation: pulseRing 1.4s ease-in-out infinite;
            position: relative;
        }
        .pulse-ring::after {
            content: ''; position: absolute; inset: 6px; border-radius: 50%;
            background: var(--accent); opacity: 0.3;
            animation: pulseCore 1.4s ease-in-out infinite;
        }
        @keyframes pulseRing {
            0%, 100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(225,29,72,0.4); }
            50% { transform: scale(1.08); box-shadow: 0 0 0 14px rgba(225,29,72,0); }
        }
        @keyframes pulseCore { 0%, 100% { opacity: 0.3; } 50% { opacity: 0.7; } }
        .baking-label { font-size: 0.75rem; color: var(--muted); letter-spacing: 0.12em; text-transform: uppercase; }

        /* Ambient glow */
        .ambient-glow {
            position: absolute; inset: -20%;
            filter: blur(80px); opacity: 0.18;
            transition: background 2s ease;
            z-index: 0; pointer-events: none;
        }

        /* Crossfade */
        @keyframes fadeIn {
            from { opacity: 0; transform: scale(1.03); }
            to   { opacity: 1; transform: scale(1); }
        }
        .img-fadein { animation: fadeIn 0.7s ease forwards; }

        /* Speed badge */
        .speed-badge {
            position: absolute; top: 14px; right: 14px;
            background: rgba(0,0,0,0.65); border: 1px solid #333;
            border-radius: 999px; padding: 3px 9px;
            font-size: 0.7rem; color: #22c55e; font-family: monospace;
            z-index: 5; backdrop-filter: blur(8px);
            animation: badgePop 0.4s cubic-bezier(0.34,1.56,0.64,1) both;
        }
        @keyframes badgePop {
            from { transform: scale(0.5) translateY(-8px); opacity: 0; }
            to   { transform: scale(1) translateY(0); opacity: 1; }
        }

        /* Combo streak */
        #comboStreak {
            position: fixed; top: 14px; left: 50%; transform: translateX(-50%);
            background: rgba(0,0,0,0.75); border: 1px solid #333;
            border-radius: 999px; padding: 5px 14px;
            font-size: 0.75rem; color: #f59e0b;
            z-index: 1002; backdrop-filter: blur(10px);
            transition: opacity 0.5s; pointer-events: none; opacity: 0;
        }
        #comboStreak.visible { opacity: 1; }

        /* Post overlay — long-press reveal */
        .post-overlay {
            position: absolute; right: 16px; bottom: 16px;
            display: flex; flex-direction: column; gap: 7px;
            z-index: 4; opacity: 0; transform: translateX(12px);
            transition: opacity 0.25s ease, transform 0.25s ease;
            pointer-events: none;
        }
        .post.overlay-visible .post-overlay { opacity: 1; transform: translateX(0); pointer-events: auto; }

        .post-btn {
            background: rgba(20,20,20,0.92); color: #fff;
            border: 1px solid #444; border-radius: 999px;
            padding: 7px 13px; cursor: pointer; font-size: 0.75rem;
            backdrop-filter: blur(8px); white-space: nowrap;
        }
        .post-btn:hover { background: rgba(50,50,50,0.95); }

        .post-meta {
            position: absolute; left: 14px; bottom: 14px; z-index: 3;
            background: rgba(10,10,10,0.82); border: 1px solid #2a2a2a;
            border-radius: 10px; padding: 9px 12px;
            max-width: min(72vw, 460px);
            font-size: 0.78rem; line-height: 1.4; color: #ccc;
            backdrop-filter: blur(12px);
            opacity: 0; transform: translateY(8px);
            transition: opacity 0.25s ease, transform 0.25s ease;
            pointer-events: none;
        }
        .post.overlay-visible .post-meta { opacity: 1; transform: translateY(0); }
        .post-meta .meta-prompt { display: -webkit-box; -webkit-line-clamp: 3; -webkit-box-orient: vertical; overflow: hidden; }

        /* Swipe hint */
        .swipe-hint {
            position: absolute; top: 50%; left: 50%;
            transform: translate(-50%, -50%);
            font-size: 1.6rem; z-index: 10;
            pointer-events: none; opacity: 0; transition: opacity 0.2s;
        }
        .swipe-hint.show { opacity: 1; }

        /* Pull-to-regen */
        #pullIndicator {
            position: fixed; top: 0; left: 0; right: 0; height: 3px;
            background: var(--accent); transform: scaleX(0); transform-origin: left;
            transition: transform 0.1s, opacity 0.3s;
            z-index: 2000; opacity: 0;
        }
        #pullIndicator.active { opacity: 1; }

        /* ===== GHOST BUTTON — tiny, top-right corner, out of the way ===== */
        #ghostBtn {
            position: fixed; top: 12px; right: 12px;
            width: 32px; height: 32px;
            background: rgba(0,0,0,0.4);
            border: 1px solid rgba(255,255,255,0.1);
            border-radius: 50%;
            color: rgba(255,255,255,0.5);
            font-size: 13px;
            cursor: pointer;
            z-index: 1003;
            display: flex; align-items: center; justify-content: center;
            backdrop-filter: blur(6px);
            transition: background 0.2s, opacity 0.2s, color 0.2s;
        }
        #ghostBtn:hover { background: rgba(40,40,40,0.7); color: #fff; }

        /* ===== PANELS ===== */
        #controls {
            position: fixed; top: 10px; left: 10px;
            background: var(--panel-bg); backdrop-filter: blur(16px);
            padding: 16px; border-radius: 14px; border: 1px solid var(--border);
            z-index: 1000; width: 340px; max-height: 90vh; overflow-y: auto;
            transition: transform 0.35s cubic-bezier(0.4,0,0.2,1), opacity 0.3s;
            box-shadow: 0 12px 40px rgba(0,0,0,0.9);
            scrollbar-width: thin; scrollbar-color: #333 transparent;
        }
        #controls.hidden { transform: translateX(-110%); opacity: 0; pointer-events: none; }

        #historyPanel {
            position: fixed; top: 10px; right: 10px;
            width: 340px; max-height: 90vh; overflow-y: auto;
            background: var(--panel-bg); backdrop-filter: blur(16px);
            padding: 16px; border-radius: 14px; border: 1px solid var(--border);
            z-index: 1000;
            transition: transform 0.35s cubic-bezier(0.4,0,0.2,1), opacity 0.3s;
            box-shadow: 0 12px 40px rgba(0,0,0,0.9);
        }
        #historyPanel.hidden { transform: translateX(110%); opacity: 0; pointer-events: none; }

        label { font-size: 0.78rem; color: var(--muted); margin-top: 10px; display: block; }
        textarea, input[type="text"], input[type="number"], select {
            width: 100%; background: #111; color: white;
            border: 1px solid #333; border-radius: 8px;
            padding: 8px; box-sizing: border-box; margin-top: 4px; font-family: inherit;
            transition: border-color 0.2s;
        }
        textarea:focus, input:focus { outline: none; border-color: var(--accent); }
        textarea { height: 64px; resize: none; }
        input[type="range"] { width: 100%; margin-top: 8px; accent-color: var(--accent); }

        #telemetry, .panel-box {
            margin-top: 12px; padding: 12px; background: #0d0d0d;
            border-radius: 8px; font-family: monospace; font-size: 0.82rem;
            border: 1px solid var(--border); color: #0ea5e9;
        }
        .stat-row { display: flex; justify-content: space-between; margin-bottom: 4px; gap: 10px; }

        details { margin-top: 10px; border: 1px solid #2a2a2a; border-radius: 8px; padding: 8px; background: #0d0d0d; }
        summary { cursor: pointer; font-size: 0.88rem; font-weight: 600; color: #ccc; user-select: none; }
        .flex-row { display: flex; gap: 10px; margin-top: 10px; }
        .flex-col { flex: 1; }

        .preset-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-top: 8px; }
        .preset-btn, .small-btn, .mode-btn {
            background: #141414; color: #ccc; border: 1px solid #333; border-radius: 8px;
            padding: 8px; cursor: pointer; font-size: 0.8rem;
            transition: background 0.15s, border-color 0.15s;
        }
        .preset-btn:hover, .small-btn:hover, .mode-btn:hover { background: #1e1e1e; border-color: #555; }
        .mode-btn.active { border-color: var(--accent); color: var(--accent); box-shadow: 0 0 0 1px var(--accent) inset; }

        button.primary {
            width: 100%; padding: 12px; background: var(--accent); color: white;
            border: none; border-radius: 8px; font-weight: 700; cursor: pointer;
            margin-top: 12px; font-size: 0.95rem; letter-spacing: 0.03em;
            transition: background 0.15s, transform 0.1s;
        }
        button.primary:hover { background: var(--accent-dark); }
        button.primary:active { transform: scale(0.97); }

        /* Bottom toggle buttons */
        .toggle-ui, .toggle-history {
            position: fixed; bottom: 18px; z-index: 1001;
            background: rgba(20,20,20,0.88); border: 1px solid #444; border-radius: 50%;
            width: 42px; height: 42px; color: white; cursor: pointer; font-size: 16px;
            backdrop-filter: blur(10px);
            transition: background 0.2s, transform 0.15s;
            display: flex; align-items: center; justify-content: center;
            box-shadow: 0 4px 16px rgba(0,0,0,0.6);
        }
        .toggle-ui:hover, .toggle-history:hover { background: rgba(40,40,40,0.95); transform: scale(1.08); }
        .toggle-ui { left: 18px; }
        .toggle-history { right: 18px; }

        /* ===== AUTO-SCROLL BAR ===== */
        #autoScrollBar {
            position: fixed; bottom: 18px; left: 50%; transform: translateX(-50%);
            background: rgba(10,10,10,0.88); border: 1px solid var(--border);
            border-radius: 999px; padding: 8px 16px;
            z-index: 1002; backdrop-filter: blur(12px);
            display: none; align-items: center; gap: 10px;
            font-size: 0.75rem; color: var(--muted); white-space: nowrap;
            box-shadow: 0 4px 20px rgba(0,0,0,0.7);
        }
        #autoScrollBar.visible { display: flex; }
        #autoScrollBar input[type="range"] { width: 90px; margin: 0; }

        /* ===== MOOD PRESETS ===== */
        .mood-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 6px; margin-top: 8px; }
        .mood-btn {
            background: #141414; color: #ccc; border: 1px solid #333; border-radius: 8px;
            padding: 7px 4px; cursor: pointer; font-size: 0.7rem; text-align: center; line-height: 1.4;
            transition: background 0.15s, border-color 0.15s;
        }
        .mood-btn:hover { background: #1e1e1e; border-color: #555; }
        .mood-btn.active { border-color: var(--accent); color: var(--accent); }
        .mood-emoji { font-size: 1rem; display: block; margin-bottom: 2px; }

        /* ===== ZEN MODE — hides EVERYTHING including autoscroll bar & ghost btn ===== */
        .zen-active #controls,
        .zen-active #historyPanel,
        .zen-active .toggle-ui,
        .zen-active .toggle-history,
        .zen-active .post-overlay,
        .zen-active .post-meta,
        .zen-active #comboStreak,
        .zen-active #autoScrollBar,
        .zen-active #ghostBtn { opacity: 0 !important; pointer-events: none !important; }

        /* ===== DARK ROOM ===== */
        .darkroom-active #controls { opacity: 0.07; transition: opacity 0.4s; }
        .darkroom-active #controls:hover { opacity: 1; }
        .darkroom-active #historyPanel { opacity: 0.07; transition: opacity 0.4s; }
        .darkroom-active #historyPanel:hover { opacity: 1; }
        .darkroom-active .toggle-ui,
        .darkroom-active .toggle-history { opacity: 0.12; }
        .darkroom-active .toggle-ui:hover,
        .darkroom-active .toggle-history:hover { opacity: 1; }

        #vignette {
            position: fixed; inset: 0;
            background: radial-gradient(ellipse at center, transparent 50%, rgba(0,0,0,0.88) 100%);
            pointer-events: none; z-index: 999; opacity: 0; transition: opacity 0.6s;
        }
        .darkroom-active #vignette { opacity: 1; }

        /* ===== HISTORY ===== */
        .history-item { border: 1px solid #222; background: #0d0d0d; border-radius: 10px; padding: 10px; margin-top: 10px; }
        .history-item img { width: 100%; aspect-ratio: 3/4; object-fit: cover; border-radius: 8px; border: 1px solid #222; background: #050505; cursor: pointer; }
        .history-actions { display: grid; grid-template-columns: 1fr 1fr; gap: 5px; margin-top: 8px; }
        .history-text { font-size: 0.78rem; color: #ccc; margin-top: 8px; line-height: 1.4; }
        .badge { display: inline-block; padding: 2px 8px; border-radius: 999px; font-size: 0.7rem; background: #1a1a1a; color: #aaa; border: 1px solid #2a2a2a; }
        .fav { color: #ffcc5a; }
        .checkbox-row { display: flex; align-items: center; gap: 8px; color: var(--text); font-size: 0.88rem; margin-top: 10px; cursor: pointer; }
        .checkbox-row input { width: auto; accent-color: var(--accent); }
        .section-title { font-size: 0.72rem; color: var(--muted); margin-top: 14px; text-transform: uppercase; letter-spacing: 0.1em; }

.lora-panel { border: 1px solid #242424; background: #0f0f10; border-radius: 14px; padding: 12px; margin-top: 10px; }
.lora-item { border: 1px solid #262626; background: #121212; border-radius: 12px; padding: 10px; margin-top: 10px; }
.lora-top { display: flex; justify-content: space-between; align-items: center; gap: 8px; flex-wrap: wrap; }
.lora-name { font-size: 0.88rem; font-weight: 700; color: var(--text); }
.lora-meta { font-size: 0.74rem; color: var(--muted); margin-top: 4px; word-break: break-all; }
.lora-actions { display: flex; align-items: center; gap: 10px; margin-top: 10px; flex-wrap: wrap; }
.lora-strength-wrap { flex: 1; min-width: 160px; }
.status-box { margin-top: 8px; min-height: 18px; font-size: 0.78rem; color: var(--muted); }
    </style>
</head>
<body id="mainBody">
    <div id="pullIndicator"></div>
    <div id="vignette"></div>
    <div id="comboStreak">🔥 <span id="comboCount">0</span> frames generated</div>

    <!-- Ghost/Zen button — tiny, top-right, out of the way -->
    <button id="ghostBtn" onclick="toggleZen(event)" title="Zen Mode">👻</button>

    <button class="toggle-ui" onclick="toggleUI()" title="Controls">👁️</button>
    <button class="toggle-history" onclick="toggleHistory()" title="History">🕘</button>

    <!-- Auto-scroll bar — centered bottom pill, hidden in zen -->
    <div id="autoScrollBar">
        ⏩ <span id="autoScrollSpeedVal">5</span>s per image
        <input type="range" id="autoScrollSpeed" min="2" max="15" value="5" oninput="updateAutoScrollSpeed()">
        <button class="small-btn" style="padding:3px 10px;font-size:0.72rem;" onclick="stopAutoScroll()">⏹</button>
    </div>

    <div id="controls">
        <h2 style="margin:0 0 12px 0;font-size:1.15rem;letter-spacing:0.02em;">🔴 Vision Studio</h2>

        <div id="telemetry">
            <div class="stat-row"><span>Buffer Ready:</span> <span id="buf_count">0</span> / <span id="buf_target">3</span></div>
            <div class="stat-row"><span>Jobs Active:</span> <span id="active_jobs">0</span> / <span id="max_jobs">2</span></div>
            <div class="stat-row"><span>Current Time:</span> <span id="current_timer">0.00s</span></div>
            <div class="stat-row"><span>Avg Time:</span> <span id="avg_timer">0.00s</span></div>
            <div class="stat-row"><span>Engine State:</span> <span id="engine_state" style="color:#94a3b8;">⏸️ Idle</span></div>
            <div class="stat-row"><span>Streak:</span> <span id="streak_display" style="color:#f59e0b;">🔥 0</span></div>
        </div>

        <div class="section-title">Mood</div>
        <div class="mood-grid">
            <button type="button" class="mood-btn active" id="mood_studio"   onclick="applyMood('studio')"><span class="mood-emoji">🎬</span>Studio</button>
            <button type="button" class="mood-btn" id="mood_lofi"     onclick="applyMood('lofi')"><span class="mood-emoji">☕</span>Lo-fi</button>
            <button type="button" class="mood-btn" id="mood_midnight" onclick="applyMood('midnight')"><span class="mood-emoji">🌙</span>Midnight</button>
            <button type="button" class="mood-btn" id="mood_dream"    onclick="applyMood('dream')"><span class="mood-emoji">✨</span>Dream</button>
            <button type="button" class="mood-btn" id="mood_cinema"   onclick="applyMood('cinema')"><span class="mood-emoji">🎞️</span>Cinema</button>
            <button type="button" class="mood-btn" id="mood_darkroom" onclick="applyMood('darkroom')"><span class="mood-emoji">🕯️</span>Dark Room</button>
            <button type="button" class="mood-btn" id="mood_neon"     onclick="applyMood('neon')"><span class="mood-emoji">⚡</span>Neon</button>
            <button type="button" class="mood-btn" id="mood_zen"      onclick="applyMood('zen')"><span class="mood-emoji">🧘</span>Zen</button>
        </div>

        <div class="section-title">Mode</div>
        <div class="preset-grid">
            <button type="button" class="mode-btn active" id="mode_balanced" onclick="applyMode('balanced')">Balanced</button>
            <button type="button" class="mode-btn" id="mode_fast"     onclick="applyMode('fast')">Fast</button>
            <button type="button" class="mode-btn" id="mode_quality"  onclick="applyMode('quality')">Quality</button>
        </div>


<div class="section-title">Custom NoobAI LoRAs</div>
<div class="lora-panel">
    <label>CivitAI API Token</label>
    <input type="password" id="civitai_token" placeholder="Paste your CivitAI token here">
    <button type="button" class="small-btn" style="width:100%;margin-top:8px;" onclick="saveCivitaiToken()">🔐 Save Token For Session</button>
    <label style="margin-top:10px;">CivitAI Direct Download Link</label>
    <input type="text" id="civitai_lora_url" placeholder="https://civitai.com/api/download/models/... or model page link">
    <label>Display Name (optional)</label>
    <input type="text" id="civitai_lora_name" placeholder="My Character LoRA">
    <button type="button" class="small-btn" style="width:100%;margin-top:10px;" onclick="addCivitaiLora()">➕ Add LoRA From CivitAI</button>
    <div id="lora_status" class="status-box">No custom LoRA loaded yet.</div>
    <div id="lora_list"></div>
</div>

        <label>Preload Buffer Size: <span id="buffer_val">3</span> frames</label>
        <input type="range" id="buffer_size" min="1" max="10" value="3" oninput="document.getElementById('buffer_val').innerText=this.value; maintainBuffer();">

        <label>Subject Prompt</label>
        <textarea id="prompt">masterpiece, best quality, newest</textarea>

        <div class="preset-grid">
            <button type="button" class="small-btn" onclick="applyPromptPreset('anime_clean')">Anime Clean</button>
            <button type="button" class="small-btn" onclick="applyPromptPreset('cinematic')">Cinematic</button>
            <button type="button" class="small-btn" onclick="applyPromptPreset('soft')">Soft Lighting</button>
            <button type="button" class="small-btn" onclick="applyPromptPreset('detail')">Detail Boost</button>
        </div>

        <div class="preset-grid">
            <button type="button" class="small-btn" onclick="mutatePrompt('detail')">+ Detail</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('dramatic')">+ Dramatic</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('pose')">+ Pose</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('camera')">+ Camera</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('style')">+ Style</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('random')">+ Random</button>
        </div>

        <label style="display:flex;align-items:center;gap:8px;color:var(--text);font-size:0.88rem;margin-top:10px;cursor:pointer;">
            <input type="checkbox" id="grunge" style="width:auto;accent-color:var(--accent);"> 🔥 Apply Glitch/Grunge
        </label>
        <label class="checkbox-row"><input type="checkbox" id="infinite_mode"> ♾️ Infinite Mode</label>
        <label class="checkbox-row"><input type="checkbox" id="seed_lock"> 🔒 Seed Lock</label>
        <label class="checkbox-row" onclick="toggleAutoScroll()"><input type="checkbox" id="auto_scroll_toggle"> 🔄 Auto-scroll</label>

        <div class="flex-row">
            <div class="flex-col">
                <label>Seed</label>
                <input type="number" id="seed" value="">
            </div>
            <div class="flex-col" style="display:flex;align-items:flex-end;">
                <button type="button" class="small-btn" style="width:100%;" onclick="rerollSeed()">🎲 Reroll</button>
            </div>
        </div>

        <details>
            <summary>⚙️ Advanced Parameters</summary>
            <label>Negative Prompt</label>
            <textarea id="neg_prompt" style="height:40px;">worst quality, low quality, normal quality, blurry, bad anatomy</textarea>

            <div class="preset-grid">
                <button type="button" class="small-btn" onclick="applyNegativePreset('default')">Default Neg</button>
                <button type="button" class="small-btn" onclick="applyNegativePreset('clean')">Cleaner Faces</button>
                <button type="button" class="small-btn" onclick="applyNegativePreset('sharp')">Sharp Output</button>
                <button type="button" class="small-btn" onclick="applyNegativePreset('nsfw')">NSFW Detail</button>
            </div>

            <label>Inference Steps: <span id="steps_val">4</span></label>
            <input type="range" id="steps" min="1" max="12" value="4" oninput="document.getElementById('steps_val').innerText=this.value">

            <label>CFG Scale: <span id="cfg_val">1.5</span></label>
            <input type="range" id="cfg" min="0.5" max="6.0" step="0.1" value="1.5" oninput="document.getElementById('cfg_val').innerText=this.value">

            <label>Resolution Presets</label>
            <div class="preset-grid">
                <button type="button" class="preset-btn" onclick="setRes(480,720)">Default 480×720</button>
                <button type="button" class="preset-btn" onclick="setRes(512,512)">1:1 512×512</button>
                <button type="button" class="preset-btn" onclick="setRes(768,1024)">3:4 768×1024</button>
                <button type="button" class="preset-btn" onclick="setRes(832,1216)">SDXL 832×1216</button>
                <button type="button" class="preset-btn" onclick="setRes(1216,832)">Wide 1216×832</button>
                <button type="button" class="preset-btn" onclick="setRes(1024,1536)">Portrait 1024×1536</button>
            </div>

            <div class="flex-row">
                <div class="flex-col"><label>Width</label><input type="number" id="width" value="480" min="256" max="2048" step="16" oninput="handleDimensionChange('width')"></div>
                <div class="flex-col"><label>Height</label><input type="number" id="height" value="720" min="256" max="2048" step="16" oninput="handleDimensionChange('height')"></div>
            </div>

            <label class="checkbox-row"><input type="checkbox" id="aspect_lock" checked> 🔗 Aspect Lock</label>
            <label class="checkbox-row"><input type="checkbox" id="vram_safe" checked> 🛡️ VRAM Safe Mode</label>
        </details>

        <button class="primary" onclick="igniteEngine()">Ignite Feed ▶</button>
        <button class="primary" style="background:#1a1a1a;border:1px solid #333;" onclick="stopInfinite()">Stop Infinite ⏹</button>
    </div>

    <div id="historyPanel" class="hidden">
        <h2 style="margin:0 0 12px 0;font-size:1.05rem;">🕘 History</h2>
        <div class="panel-box">
            <div class="stat-row"><span>Saved Frames:</span><span id="history_count">0</span></div>
            <div class="stat-row"><span>Favorites:</span><span id="favorite_count">0</span></div>
        </div>
        <div id="historyList"></div>
    </div>

    <div id="feed" onscroll="handleScroll()"></div>

    <script>
        // ===== STATE =====
        let activeJobs = 0;
        const MAX_ACTIVE_JOBS = 2;
        const DOM_PRUNE_LIMIT = 25;
        let generationDurations = [];
        let liveTimers = {};
        let historyItems = [];
        let lastSeedUsed = null;
        let aspectRatio = 480 / 720;
        let infiniteStopped = false;
        let comboCount = 0;
        let comboTimeout = null;
        let autoScrollInterval = null;
        let autoScrollActive = false;
        let currentMood = 'studio';


let customLoras = [];
let loraRefreshInFlight = false;

async function saveCivitaiToken(){
    const token = document.getElementById('civitai_token').value.trim();
    if(!token){
        setLoraStatus('Paste your CivitAI token first.', true);
        return;
    }
    setLoraStatus('Saving CivitAI token for this session...');
    try{
        const res = await fetch('/api/lora/token', {
            method:'POST',
            headers:{'Content-Type':'application/json'},
            body: JSON.stringify({token})
        });
        const data = await res.json();
        if(!res.ok || !data.ok) throw new Error(data.error || 'Could not save token');
        setLoraStatus('CivitAI token saved for this session.');
    }catch(err){
        setLoraStatus(err.message || 'Could not save token.', true);
    }
}

async function refreshLoraList(){
    if(loraRefreshInFlight) return;
    loraRefreshInFlight = true;
    try{
        const res = await fetch('/api/lora/list?_t='+Date.now(), {cache:'no-store'});
        const data = await res.json();
        customLoras = Array.isArray(data.items) ? data.items : [];
        renderLoraList();
    }catch(err){
        setLoraStatus('Could not refresh custom LoRA list.', true);
    }finally{
        loraRefreshInFlight = false;
    }
}

function setLoraStatus(message, isError=false){
    const el = document.getElementById('lora_status');
    el.textContent = message || '';
    el.style.color = isError ? '#f87171' : 'var(--muted)';
}

function renderLoraList(){
    const wrap = document.getElementById('lora_list');
    wrap.innerHTML = '';
    if(!customLoras.length){
        wrap.innerHTML = '<div class="lora-meta">Add a direct CivitAI download link to load a custom NoobAI LoRA into this session.</div>';
        return;
    }
    customLoras.forEach(item=>{
        const card = document.createElement('div');
        card.className = 'lora-item';
        const checked = item.enabled ? 'checked' : '';
        const strength = Number(item.strength ?? 1).toFixed(2);
        card.innerHTML = `
            <div class="lora-top">
                <div>
                    <div class="lora-name">${escapeHtml(item.name || item.filename || 'Custom LoRA')}</div>
                    <div class="lora-meta">${escapeHtml(item.filename || '')}</div>
                </div>
                <label class="checkbox-row" style="margin-top:0;"><input type="checkbox" ${checked} onchange="toggleCustomLora('${item.key}', this.checked)"> Enabled</label>
            </div>
            <div class="lora-actions">
                <div class="lora-strength-wrap">
                    <label>LoRA Strength: <span id="lora_strength_val_${item.key}">${strength}</span></label>
                    <input type="range" min="0" max="2" step="0.05" value="${item.strength ?? 1}" oninput="document.getElementById('lora_strength_val_${item.key}').innerText=this.value" onchange="updateCustomLoraStrength('${item.key}', this.value)">
                </div>
            </div>
        `;
        wrap.appendChild(card);
    });
}

function escapeHtml(text){
    return String(text ?? '').replace(/[&<>"']/g, m => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[m]));
}

async function addCivitaiLora(){
    const url = document.getElementById('civitai_lora_url').value.trim();
    const name = document.getElementById('civitai_lora_name').value.trim();
    const token = document.getElementById('civitai_token').value.trim();
    if(!url){
        setLoraStatus('Paste a CivitAI direct download link first.', true);
        return;
    }
    setLoraStatus('Downloading and attaching LoRA...');
    try{
        const res = await fetch('/api/lora/add', {
            method:'POST',
            headers:{'Content-Type':'application/json'},
            body: JSON.stringify({url, name, token})
        });
        const data = await res.json();
        if(!res.ok || !data.ok) throw new Error(data.error || 'Failed to add LoRA');
        document.getElementById('civitai_lora_url').value = '';
        if(!name) document.getElementById('civitai_lora_name').value = '';
        setLoraStatus(`Loaded: ${data.item.name}`);
        await refreshLoraList();
fetch('/api/lora/token').then(r=>r.json()).then(data=>{
    if(data && data.ok && data.has_token){
        const el = document.getElementById('civitai_token');
        if(el) el.placeholder = 'Token saved for this session';
    }
}).catch(()=>{});
    }catch(err){
        setLoraStatus(err.message || 'Could not add LoRA.', true);
    }
}

async function toggleCustomLora(key, enabled){
    try{
        const res = await fetch('/api/lora/update', {
            method:'POST',
            headers:{'Content-Type':'application/json'},
            body: JSON.stringify({key, enabled})
        });
        const data = await res.json();
        if(!res.ok || !data.ok) throw new Error(data.error || 'Toggle failed');
        setLoraStatus((data.item?.name || 'LoRA') + (enabled ? ' enabled.' : ' disabled.'));
        await refreshLoraList();
    }catch(err){
        setLoraStatus(err.message || 'Could not update LoRA.', true);
        await refreshLoraList();
    }
}

async function updateCustomLoraStrength(key, strength){
    try{
        const res = await fetch('/api/lora/update', {
            method:'POST',
            headers:{'Content-Type':'application/json'},
            body: JSON.stringify({key, strength: parseFloat(strength)})
        });
        const data = await res.json();
        if(!res.ok || !data.ok) throw new Error(data.error || 'Strength update failed');
        setLoraStatus(`${data.item?.name || 'LoRA'} strength set to ${Number(data.item?.strength ?? strength).toFixed(2)}.`);
    }catch(err){
        setLoraStatus(err.message || 'Could not change LoRA strength.', true);
        await refreshLoraList();
    }
}

        // ===== PRESETS =====
        const promptPresets = {
            anime_clean: "masterpiece, best quality, newest, clean lineart, polished anime shading, expressive eyes",
            cinematic:   "masterpiece, best quality, newest, cinematic lighting, dramatic shadows, filmic composition",
            soft:        "masterpiece, best quality, newest, soft lighting, pastel glow, delicate atmosphere",
            detail:      "masterpiece, best quality, newest, ultra detailed, intricate details, refined textures"
        };
        const negativePresets = {
            default: "worst quality, low quality, normal quality, blurry, bad anatomy",
            clean:   "worst quality, low quality, blurry, cross-eye, deformed eyes, bad anatomy, bad hands, extra fingers",
            sharp:   "low quality, blurry, soft focus, foggy details, smudged lines, jpeg artifacts",
            nsfw:    "worst quality, low quality, blurry, deformed anatomy, bad hands, extra limbs"
        };
        const mutations = {
            detail:    "intricate details, refined textures, sharp focus",
            dramatic:  "dramatic lighting, intense contrast, striking composition",
            pose:      "dynamic pose, expressive body language, natural gesture",
            camera:    "low angle shot, cinematic framing, depth of field",
            style:     "stylized rendering, polished finish, premium anime aesthetic",
            random:    ["wind-swept hair, motion energy","golden hour lighting, warm glow","rainy atmosphere, reflective surfaces","neon rim light, night city vibe","soft bloom, dreamy mood","heroic stance, centered framing"]
        };

        // ===== MOOD SYSTEM =====
        const moods = {
            studio:   { prompt: "", steps: 4, cfg: 1.5, w: 480, h: 720, accentColor: "#e11d48", darkroom: false },
            lofi:     { prompt: "lo-fi aesthetic, muted colors, film grain, soft vignette", steps: 4, cfg: 1.5, w: 480, h: 720, accentColor: "#a78bfa", darkroom: false },
            midnight: { prompt: "midnight, dark atmosphere, moonlight, deep shadows, blue hour", steps: 5, cfg: 1.8, w: 480, h: 720, accentColor: "#3b82f6", darkroom: false },
            dream:    { prompt: "dreamlike, soft glow, ethereal, surreal, pastel haze, magical", steps: 5, cfg: 1.6, w: 480, h: 720, accentColor: "#f0abfc", darkroom: false },
            cinema:   { prompt: "cinematic, film grain, letterbox, dramatic lighting, anamorphic lens flare", steps: 6, cfg: 2.0, w: 768, h: 432, accentColor: "#f59e0b", darkroom: false },
            darkroom: { prompt: "", steps: 4, cfg: 1.5, w: 480, h: 720, accentColor: "#6b7280", darkroom: true },
            neon:     { prompt: "neon lights, cyberpunk, night city, glowing edges, rain reflections", steps: 5, cfg: 1.8, w: 480, h: 720, accentColor: "#22d3ee", darkroom: false },
            zen:      { prompt: "serene, minimalist, soft natural light, tranquil, breathing room", steps: 4, cfg: 1.5, w: 480, h: 720, accentColor: "#86efac", darkroom: false }
        };

        function applyMood(name) {
            const m = moods[name]; if (!m) return;
            currentMood = name;
            document.querySelectorAll('.mood-btn').forEach(b => b.classList.remove('active'));
            document.getElementById('mood_' + name).classList.add('active');
            document.documentElement.style.setProperty('--accent', m.accentColor);
            document.documentElement.style.setProperty('--accent-dark', shadeColor(m.accentColor, -20));
            const promptEl = document.getElementById('prompt');
            if (m.prompt && !promptEl.value.includes(m.prompt.split(',')[0])) {
                promptEl.value = 'masterpiece, best quality, newest' + (m.prompt ? ', ' + m.prompt : '');
            }
            document.getElementById('steps').value = m.steps; document.getElementById('steps_val').innerText = m.steps;
            document.getElementById('cfg').value = m.cfg; document.getElementById('cfg_val').innerText = m.cfg;
            setRes(m.w, m.h);
            document.body.classList.toggle('darkroom-active', m.darkroom);
        }

        function shadeColor(color, percent) {
            const num = parseInt(color.replace('#',''), 16), amt = Math.round(2.55 * percent);
            const R = Math.max(0, Math.min(255, (num >> 16) + amt));
            const G = Math.max(0, Math.min(255, ((num >> 8) & 0x00FF) + amt));
            const B = Math.max(0, Math.min(255, (num & 0x0000FF) + amt));
            return '#' + (0x1000000 + R*0x10000 + G*0x100 + B).toString(16).slice(1);
        }

        // ===== COMBO STREAK =====
        function bumpCombo() {
            comboCount++;
            const el = document.getElementById('comboStreak');
            document.getElementById('comboCount').innerText = comboCount;
            document.getElementById('streak_display').innerText = '🔥 ' + comboCount;
            el.classList.add('visible');
            if (comboTimeout) clearTimeout(comboTimeout);
            comboTimeout = setTimeout(() => el.classList.remove('visible'), 4000);
        }

        // ===== AUTO-SCROLL — seconds-based =====
        function toggleAutoScroll() {
            const checkbox = document.getElementById('auto_scroll_toggle');
            // Slight delay so the label click registers the checkbox state first
            setTimeout(() => {
                autoScrollActive = checkbox.checked;
                const bar = document.getElementById('autoScrollBar');
                if (autoScrollActive) { bar.classList.add('visible'); startAutoScroll(); }
                else { bar.classList.remove('visible'); stopAutoScrollInterval(); }
            }, 0);
        }

        function startAutoScroll() {
            stopAutoScrollInterval();
            if (!autoScrollActive) return;
            const seconds = parseInt(document.getElementById('autoScrollSpeed').value); // 2–15 seconds
            autoScrollInterval = setInterval(() => {
                const feed = document.getElementById('feed');
                const currentIndex = Math.round(feed.scrollTop / window.innerHeight);
                feed.scrollTo({ top: (currentIndex + 1) * window.innerHeight, behavior: 'smooth' });
            }, seconds * 1000);
        }

        function stopAutoScrollInterval() {
            if (autoScrollInterval) { clearInterval(autoScrollInterval); autoScrollInterval = null; }
        }

        function stopAutoScroll() {
            autoScrollActive = false;
            document.getElementById('auto_scroll_toggle').checked = false;
            document.getElementById('autoScrollBar').classList.remove('visible');
            stopAutoScrollInterval();
        }

        function updateAutoScrollSpeed() {
            const val = document.getElementById('autoScrollSpeed').value;
            document.getElementById('autoScrollSpeedVal').innerText = val;
            if (autoScrollActive) startAutoScroll(); // restart timer with new interval
        }

        // ===== DOMINANT COLOR AMBIENT =====
        function extractDominantColor(img) {
            try {
                const canvas = document.createElement('canvas');
                canvas.width = 32; canvas.height = 32;
                const ctx = canvas.getContext('2d');
                ctx.drawImage(img, 0, 0, 32, 32);
                const data = ctx.getImageData(0, 0, 32, 32).data;
                let r=0, g=0, b=0, count=0;
                for (let i=0; i<data.length; i+=16) { r+=data[i]; g+=data[i+1]; b+=data[i+2]; count++; }
                return `rgb(${Math.round(r/count)},${Math.round(g/count)},${Math.round(b/count)})`;
            } catch(e) { return '#222'; }
        }

        // ===== PULL-TO-REGENERATE =====
        let pullStartY = 0, isPulling = false;
        document.getElementById('feed').addEventListener('touchstart', (e) => {
            if (document.getElementById('feed').scrollTop === 0) { pullStartY = e.touches[0].clientY; isPulling = true; }
        }, { passive: true });
        document.getElementById('feed').addEventListener('touchmove', (e) => {
            if (!isPulling) return;
            const delta = e.touches[0].clientY - pullStartY;
            if (delta > 0) {
                const ind = document.getElementById('pullIndicator');
                ind.style.transform = `scaleX(${Math.min(delta/120, 1)})`;
                ind.classList.add('active');
            }
        }, { passive: true });
        document.getElementById('feed').addEventListener('touchend', () => {
            if (!isPulling) return; isPulling = false;
            const ind = document.getElementById('pullIndicator');
            const progress = parseFloat(ind.style.transform.replace('scaleX(','').replace(')','')) || 0;
            if (progress >= 0.99) {
                ind.style.transform = 'scaleX(1)';
                setTimeout(() => { ind.classList.remove('active'); ind.style.transform='scaleX(0)'; igniteEngine(); }, 300);
            } else { ind.classList.remove('active'); ind.style.transform='scaleX(0)'; }
        }, { passive: true });

        // ===== SWIPE LEFT/RIGHT =====
        let swipeTouchStartX=0, swipeTouchStartY=0;
        document.getElementById('feed').addEventListener('touchstart', (e) => {
            swipeTouchStartX = e.touches[0].clientX; swipeTouchStartY = e.touches[0].clientY;
        }, { passive: true });
        document.getElementById('feed').addEventListener('touchend', (e) => {
            const dx = e.changedTouches[0].clientX - swipeTouchStartX;
            const dy = e.changedTouches[0].clientY - swipeTouchStartY;
            if (Math.abs(dx) > Math.abs(dy) && Math.abs(dx) > 60) {
                const feed = document.getElementById('feed');
                const currentIndex = Math.round(feed.scrollTop / window.innerHeight);
                const currentPost = feed.children[currentIndex];
                const hint = currentPost ? currentPost.querySelector('.swipe-hint') : null;
                if (dx < 0) {
                    if (hint) { hint.textContent='🔀 Variation'; hint.classList.add('show'); setTimeout(()=>hint.classList.remove('show'),700); }
                    document.getElementById('seed_lock').checked=true; document.getElementById('seed').value=lastSeedUsed; bakeNext();
                } else {
                    if (hint) { hint.textContent='✨ New'; hint.classList.add('show'); setTimeout(()=>hint.classList.remove('show'),700); }
                    document.getElementById('seed_lock').checked=false; bakeNext();
                }
            }
        }, { passive: true });

        // ===== LONG-PRESS =====
        let longPressTimer = null;
        function setupLongPress(post) {
            const start = () => { longPressTimer = setTimeout(() => { document.querySelectorAll('.post').forEach(p=>p.classList.remove('overlay-visible')); post.classList.toggle('overlay-visible'); }, 450); };
            const end = () => clearTimeout(longPressTimer);
            post.addEventListener('touchstart', start, { passive: true });
            post.addEventListener('touchend', end, { passive: true });
            post.addEventListener('mousedown', start);
            post.addEventListener('mouseup', end);
            post.addEventListener('mouseleave', end);
        }

        // ===== KEYBOARD SHORTCUTS =====
        document.addEventListener('keydown', (e) => {
            if (e.target.tagName==='TEXTAREA'||e.target.tagName==='INPUT') return;
            const feed = document.getElementById('feed');
            const currentIndex = Math.round(feed.scrollTop / window.innerHeight);
            if (e.key===' '||e.key==='ArrowDown') { e.preventDefault(); feed.scrollTo({top:(currentIndex+1)*window.innerHeight,behavior:'smooth'}); }
            else if (e.key==='ArrowUp') { e.preventDefault(); feed.scrollTo({top:(currentIndex-1)*window.innerHeight,behavior:'smooth'}); }
            else if (e.key==='g'||e.key==='G') bakeNext();
            else if (e.key==='f'||e.key==='F') { const p=feed.children[currentIndex]; if(p){const b=p.querySelector('.post-btn');if(b)b.click();} }
            else if (e.key==='d'||e.key==='D') { const p=feed.children[currentIndex]; if(p){const bs=p.querySelectorAll('.post-btn');if(bs[1])bs[1].click();} }
            else if (e.key==='Escape') document.querySelectorAll('.post').forEach(p=>p.classList.remove('overlay-visible'));
        });

        // ===== UI TOGGLES =====
        function toggleUI() { document.getElementById('controls').classList.toggle('hidden'); }
        function toggleHistory() { document.getElementById('historyPanel').classList.toggle('hidden'); }
        function toggleZen(e) {
            if(e) e.stopPropagation();
            document.body.classList.toggle('zen-active');
        }

        document.addEventListener('click', (e) => {
            if (document.body.classList.contains('zen-active') && !e.target.closest('#ghostBtn')) toggleZen();
            if (!e.target.closest('.post')) document.querySelectorAll('.post').forEach(p=>p.classList.remove('overlay-visible'));
        });
        document.getElementById('controls').addEventListener('click', e=>e.stopPropagation());
        document.getElementById('historyPanel').addEventListener('click', e=>e.stopPropagation());

        // ===== MODE =====
        function setModeButton(mode) {
            ["fast","balanced","quality"].forEach(m=>document.getElementById("mode_"+m).classList.remove("active"));
            document.getElementById("mode_"+mode).classList.add("active");
        }
        function applyMode(mode) {
            setModeButton(mode);
            if (mode==="fast")     { document.getElementById("steps").value=3; document.getElementById("steps_val").innerText="3"; document.getElementById("cfg").value=1.2; document.getElementById("cfg_val").innerText="1.2"; setRes(480,720); }
            else if (mode==="balanced") { document.getElementById("steps").value=4; document.getElementById("steps_val").innerText="4"; document.getElementById("cfg").value=1.5; document.getElementById("cfg_val").innerText="1.5"; setRes(480,720); }
            else if (mode==="quality")  { document.getElementById("steps").value=6; document.getElementById("steps_val").innerText="6"; document.getElementById("cfg").value=2.0; document.getElementById("cfg_val").innerText="2.0"; setRes(768,1024); }
        }

        // ===== PROMPT HELPERS =====
        function applyPromptPreset(kind) { document.getElementById("prompt").value=promptPresets[kind]||document.getElementById("prompt").value; }
        function applyNegativePreset(kind) { document.getElementById("neg_prompt").value=negativePresets[kind]||document.getElementById("neg_prompt").value; }
        function mutatePrompt(kind) {
            const box=document.getElementById("prompt");
            let extra=kind==="random"?mutations.random[Math.floor(Math.random()*mutations.random.length)]:mutations[kind]||"";
            if(!extra) return;
            box.value=box.value.trim().replace(/,\s*$/,"")+", "+extra;
        }

        // ===== SEED =====
        function rerollSeed() { const s=Math.floor(Math.random()*2147483647); document.getElementById("seed").value=s; lastSeedUsed=s; }
        function getSelectedSeed() {
            const lock=document.getElementById("seed_lock").checked;
            const seedInput=document.getElementById("seed");
            let seedVal=seedInput.value.trim();
            if(lock){if(!seedVal){seedVal=String(Math.floor(Math.random()*2147483647));seedInput.value=seedVal;}return parseInt(seedVal);}
            return -1;
        }

        // ===== RESOLUTION =====
        function setRes(w,h){document.getElementById('width').value=w;document.getElementById('height').value=h;aspectRatio=w/h;applyVramSafeIfNeeded();}
        function roundTo16(v){return Math.max(256,Math.round(v/16)*16);}
        function handleDimensionChange(changed){
            let w=parseInt(document.getElementById('width').value)||480;
            let h=parseInt(document.getElementById('height').value)||720;
            if(document.getElementById("aspect_lock").checked){
                if(changed==="width"){h=roundTo16(w/aspectRatio);document.getElementById('height').value=h;}
                else{w=roundTo16(h*aspectRatio);document.getElementById('width').value=w;}
            }else{if(w>0&&h>0)aspectRatio=w/h;}
            applyVramSafeIfNeeded();
        }
        function getSafeRes(){
            let w=parseInt(document.getElementById('width').value)||480;
            let h=parseInt(document.getElementById('height').value)||720;
            w=Math.max(256,Math.min(w,2048));h=Math.max(256,Math.min(h,2048));w=roundTo16(w);h=roundTo16(h);
            if(document.getElementById("vram_safe").checked){const max=786432;if(w*h>max){const sc=Math.sqrt(max/(w*h));w=roundTo16(w*sc);h=roundTo16(h*sc);}}
            return{w,h};
        }
        function applyVramSafeIfNeeded(){const{w,h}=getSafeRes();document.getElementById('width').value=w;document.getElementById('height').value=h;}

        // ===== TELEMETRY =====
        function updateTelemetry(){
            const feed=document.getElementById('feed');
            const currentIndex=feed.children.length>0?Math.round(feed.scrollTop/window.innerHeight):0;
            const imagesAhead=Math.max(0,feed.children.length-currentIndex-1);
            const targetBuffer=parseInt(document.getElementById('buffer_size').value);
            document.getElementById('buf_count').innerText=imagesAhead;
            document.getElementById('buf_target').innerText=targetBuffer;
            document.getElementById('active_jobs').innerText=activeJobs;
            document.getElementById('max_jobs').innerText=MAX_ACTIVE_JOBS;
            let currentLive=0;
            for(const key in liveTimers)currentLive=Math.max(currentLive,(performance.now()-liveTimers[key])/1000);
            document.getElementById("current_timer").innerText=currentLive.toFixed(2)+"s";
            const avg=generationDurations.length?generationDurations.reduce((a,b)=>a+b,0)/generationDurations.length:0;
            document.getElementById("avg_timer").innerText=avg.toFixed(2)+"s";
            const engineState=document.getElementById('engine_state');
            if(activeJobs>0)engineState.innerHTML='<span style="color:#ef4444;">🔴 Baking...</span>';
            else if(document.getElementById("infinite_mode").checked&&!infiniteStopped)engineState.innerHTML='<span style="color:#22c55e;">♾️ Infinite Ready</span>';
            else engineState.innerHTML='<span style="color:#22c55e;">🟢 Ready</span>';
            document.getElementById("history_count").innerText=historyItems.length;
            document.getElementById("favorite_count").innerText=historyItems.filter(x=>x.favorite).length;
            return{imagesAhead,targetBuffer};
        }

        // ===== HISTORY =====
        function copyText(text){navigator.clipboard.writeText(text).catch(()=>{});}
        function buildSettingsText(item){return JSON.stringify({prompt:item.prompt,negative_prompt:item.neg_prompt,steps:item.steps,cfg:item.cfg,width:item.width,height:item.height,seed:item.seed,grunge:item.grunge},null,2);}
        function downloadBlobUrl(url,filename="vision_frame.jpeg"){const a=document.createElement("a");a.href=url;a.download=filename;document.body.appendChild(a);a.click();a.remove();}
        function rerunFromHistory(item){
            document.getElementById("prompt").value=item.prompt;document.getElementById("neg_prompt").value=item.neg_prompt;
            document.getElementById("steps").value=item.steps;document.getElementById("steps_val").innerText=item.steps;
            document.getElementById("cfg").value=item.cfg;document.getElementById("cfg_val").innerText=item.cfg;
            document.getElementById("grunge").checked=item.grunge;
            document.getElementById("seed_lock").checked=true;document.getElementById("seed").value=item.seed;
            setRes(item.width,item.height);igniteEngine();
        }
        function toggleFavorite(index){historyItems[index].favorite=!historyItems[index].favorite;renderHistory();}
        function renderHistory(){
            const wrap=document.getElementById("historyList");wrap.innerHTML="";
            historyItems.slice().reverse().forEach((item,idx)=>{
                const realIndex=historyItems.length-1-idx;
                const div=document.createElement("div");div.className="history-item";
                div.innerHTML=`<img src="${item.url}" alt="history" onclick="window.open('${item.url}')"><div class="history-text"><div style="display:flex;justify-content:space-between;align-items:center;gap:6px;flex-wrap:wrap;"><span class="badge">${item.width}×${item.height}</span><span class="badge">${item.steps}s</span><span class="badge">CFG${item.cfg}</span><span class="badge">#${item.seed}</span></div><div style="margin-top:6px;">${item.prompt.slice(0,120)}</div></div><div class="history-actions"><button class="small-btn" onclick="toggleFavorite(${realIndex})">${item.favorite?'★ Unfav':'☆ Fav'}</button><button class="small-btn" onclick="downloadBlobUrl('${item.url}','vision_${realIndex}.jpeg')">⬇ Save</button><button class="small-btn" onclick='copyText(historyItems[${realIndex}].prompt)'>📋 Prompt</button><button class="small-btn" onclick='rerunFromHistory(historyItems[${realIndex}])'>🔁 Rerun</button></div>`;
                wrap.appendChild(div);
            });
            updateTelemetry();
        }

        // ===== POST ACTIONS =====
        function attachPostActions(post,item){
            const hint=document.createElement('div');hint.className='swipe-hint';post.appendChild(hint);
            const overlay=document.createElement("div");overlay.className="post-overlay";
            const favBtn=document.createElement("button");favBtn.className="post-btn";favBtn.textContent=item.favorite?"★ Fav":"☆ Fav";
            favBtn.onclick=()=>{item.favorite=!item.favorite;favBtn.textContent=item.favorite?"★ Fav":"☆ Fav";renderHistory();};
            const dlBtn=document.createElement("button");dlBtn.className="post-btn";dlBtn.textContent="⬇ Download";
            dlBtn.onclick=()=>downloadBlobUrl(item.url,`vision_${item.seed}.jpeg`);
            const cpBtn=document.createElement("button");cpBtn.className="post-btn";cpBtn.textContent="📋 Prompt";
            cpBtn.onclick=()=>copyText(item.prompt);
            const csBtn=document.createElement("button");csBtn.className="post-btn";csBtn.textContent="⚙ Settings";
            csBtn.onclick=()=>copyText(buildSettingsText(item));
            overlay.appendChild(favBtn);overlay.appendChild(dlBtn);overlay.appendChild(cpBtn);overlay.appendChild(csBtn);
            post.appendChild(overlay);
            const meta=document.createElement("div");meta.className="post-meta";
            meta.innerHTML=`<div class="meta-prompt">${item.prompt}</div><div style="margin-top:5px;color:#777;font-size:0.72rem;">${item.width}×${item.height} · ${item.steps} steps · CFG ${item.cfg} · Seed ${item.seed}</div>`;
            post.appendChild(meta);
            setupLongPress(post);
        }

        // ===== CORE GENERATION =====
        async function bakeNext(){
            if(activeJobs>=MAX_ACTIVE_JOBS)return;
            activeJobs+=1;
            const jobId="job_"+Date.now()+"_"+Math.random().toString(36).slice(2);
            liveTimers[jobId]=performance.now();
            updateTelemetry();

            const promptRaw=document.getElementById('prompt').value;
            const negPromptRaw=document.getElementById('neg_prompt').value;
            const prompt=encodeURIComponent(promptRaw);
            const neg_prompt=encodeURIComponent(negPromptRaw);
            const grunge=document.getElementById('grunge').checked;
            const steps=document.getElementById('steps').value;
            const cfg=document.getElementById('cfg').value;
            const{w,h}=getSafeRes();
            const seed=getSelectedSeed();

            const post=document.createElement('div');post.className='post';
            const placeholder=document.createElement('div');placeholder.className='blur-placeholder';
            placeholder.innerHTML='<div class="pulse-ring"></div><div class="baking-label">Baking frame...</div>';
            post.appendChild(placeholder);
            const glow=document.createElement('div');glow.className='ambient-glow';post.appendChild(glow);
            const feed=document.getElementById('feed');feed.appendChild(post);

            try{
                const url=`/api/generate?prompt=${prompt}&neg_prompt=${neg_prompt}&grunge=${grunge}&steps=${steps}&cfg=${cfg}&width=${w}&height=${h}&seed=${seed}&_t=${Date.now()}`;
                const response=await fetch(url,{cache:'no-store'});
                if(!response.ok)throw new Error('Failed');
                const blob=await response.blob();
                const imgUrl=URL.createObjectURL(blob);
                const usedSeed=parseInt(response.headers.get("X-Seed")||seed||0);
                lastSeedUsed=usedSeed;
                if(document.getElementById("seed_lock").checked)document.getElementById("seed").value=usedSeed;

                const imgWrap=document.createElement('div');imgWrap.className='post-img-wrap';
                const img=document.createElement("img");
                img.src=imgUrl;img.loading="eager";img.decoding="async";img.className='img-fadein';
                img.ondblclick=()=>window.open(imgUrl,'_blank');
                img.onload=()=>{
                    try{
                        const color=extractDominantColor(img);
                        glow.style.background=color;
                        post.style.background=`radial-gradient(ellipse at center,${color}22 0%,#000 100%)`;
                    }catch(e){}
                    placeholder.style.transition='opacity 0.5s';
                    placeholder.style.opacity='0';
                    setTimeout(()=>{if(placeholder.parentNode===post)post.removeChild(placeholder);},500);
                };
                imgWrap.appendChild(img);post.appendChild(imgWrap);

                const duration=(performance.now()-liveTimers[jobId])/1000;
                generationDurations.push(duration);
                if(generationDurations.length>20)generationDurations.shift();

                const badge=document.createElement('div');badge.className='speed-badge';
                badge.textContent=`⚡ ${duration.toFixed(1)}s`;
                post.appendChild(badge);
                setTimeout(()=>{badge.style.transition='opacity 1.5s';badge.style.opacity='0';},3000);

                const item={url:imgUrl,prompt:promptRaw,neg_prompt:negPromptRaw,steps:parseInt(steps),cfg:parseFloat(cfg),width:w,height:h,seed:usedSeed,grunge:grunge,favorite:false};
                historyItems.push(item);renderHistory();attachPostActions(post,item);
                bumpCombo();

            }catch(err){
                placeholder.innerHTML='<div style="font-size:1.8rem;">⚠️</div><div class="baking-label" style="color:#e11d48;">Engine Stall · press G to retry</div>';
            }finally{
                delete liveTimers[jobId];
                if(feed.children.length>DOM_PRUNE_LIMIT){
                    const currentIndex=Math.round(feed.scrollTop/window.innerHeight);
                    if(currentIndex>5){
                        const removed=feed.removeChild(feed.firstChild);
                        const oldImg=removed.querySelector('img');
                        if(oldImg?.src?.startsWith('blob:'))URL.revokeObjectURL(oldImg.src);
                    }
                }
                activeJobs-=1;
                maintainBuffer();
                if(document.getElementById("infinite_mode").checked&&!infiniteStopped)setTimeout(()=>maintainBuffer(),10);
            }
        }

        function maintainBuffer(){const stats=updateTelemetry();while((stats.imagesAhead+activeJobs)<stats.targetBuffer&&activeJobs<MAX_ACTIVE_JOBS){bakeNext();}}
        function handleScroll(){maintainBuffer();}
        function igniteEngine(){infiniteStopped=false;document.getElementById('feed').innerHTML='';activeJobs=0;liveTimers={};comboCount=0;document.getElementById('comboCount').innerText='0';maintainBuffer();}
        function stopInfinite(){infiniteStopped=true;document.getElementById("infinite_mode").checked=false;updateTelemetry();}

        // ===== INIT =====
        setInterval(updateTelemetry,200);
        applyMode('balanced');
        applyNegativePreset('default');
        document.getElementById("cfg").value=1.5;document.getElementById("cfg_val").innerText="1.5";
        setRes(480,720);
        applyMood('studio');
        refreshLoraList();
    </script>
</body>
</html>
"""


# --- CUSTOM LORA BACKEND HELPERS ---
def _slugify(value: str) -> str:
    value = re.sub(r'[^a-zA-Z0-9]+', '-', (value or '').strip().lower()).strip('-')
    return value or 'custom-lora'

def _main_list(name):
    obj = getattr(__main__, name, None)
    return obj if isinstance(obj, list) else []

def _save_registry():
    saver = getattr(__main__, 'save_custom_lora_registry', None)
    if callable(saver):
        saver()

def _get_model():
    return getattr(__main__, 'loaded_model', None)

CIVITAI_TOKEN_SESSION = getattr(__main__, 'CIVITAI_TOKEN_SESSION', '')

def _get_civitai_token():
    return getattr(__main__, 'CIVITAI_TOKEN_SESSION', '') or ''

def _set_civitai_token(token: str):
    token = (token or '').strip()
    setattr(__main__, 'CIVITAI_TOKEN_SESSION', token)
    return token

def _custom_registry():
    return getattr(__main__, 'CUSTOM_LORA_REGISTRY', [])

def _custom_dir():
    return getattr(__main__, 'CUSTOM_LORA_DIR', '/content/custom_noobai_loras')

def _normalize_civitai_url(raw_url: str, token: str = '') -> str:
    raw_url = (raw_url or '').strip()
    token = (token or '').strip()
    parsed = urllib.parse.urlparse(raw_url)
    host = (parsed.netloc or '').lower()
    qs = urllib.parse.parse_qs(parsed.query)
    if 'civitai' in host and '/models/' in parsed.path and qs.get('modelVersionId'):
        version_id = qs['modelVersionId'][0]
        token = token or qs.get('token', [None])[0]
        direct = f'https://civitai.com/api/download/models/{version_id}'
        if token:
            direct += '?' + urllib.parse.urlencode({'token': token})
        return direct
    if token and 'civitai' in host and '/api/download/models/' in parsed.path and not qs.get('token'):
        merged_qs = {k: v[0] for k, v in qs.items()}
        merged_qs['token'] = token
        merged = urllib.parse.urlencode(merged_qs)
        return urllib.parse.urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, merged, parsed.fragment))
    return raw_url

def _download_file(url: str, output_dir: str, token: str = ''):
    os.makedirs(output_dir, exist_ok=True)
    token = (token or '').strip()
    headers = {'User-Agent': 'Mozilla/5.0'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as resp:
        final_url = resp.geturl()
        headers = resp.info()
        disposition = headers.get('Content-Disposition', '')
        filename = None
        match = re.search(r'filename="?([^"]+)"?', disposition)
        if match:
            filename = match.group(1)
        if not filename:
            filename = os.path.basename(urllib.parse.urlparse(final_url).path) or f'lora_{int(time.time())}.safetensors'
        if '.' not in os.path.basename(filename):
            filename = filename + '.safetensors'
        target = os.path.join(output_dir, filename)
        with open(target, 'wb') as f:
            f.write(resp.read())
    return target, filename, final_url

def _apply_active_loras(model):
    names = ['lightning_base']
    weights = [1.0]
    for item in _custom_registry():
        if item.get('enabled') and item.get('loaded'):
            names.append(item['adapter_name'])
            weights.append(float(item.get('strength', 1.0)))
    model.set_adapters(names, adapter_weights=weights)

@app.get("/api/lora/token")
def civitai_token_status():
    return JSONResponse({'ok': True, 'has_token': bool(_get_civitai_token())})

@app.post("/api/lora/token")
async def save_civitai_token_api(payload: dict = Body(...)):
    token = (payload.get('token') or '').strip()
    if not token:
        return JSONResponse({'ok': False, 'error': 'Missing CivitAI token.'}, status_code=400)
    _set_civitai_token(token)
    return JSONResponse({'ok': True, 'has_token': True})

@app.get("/api/lora/list")
def list_loras():
    items = []
    for item in _custom_registry():
        items.append({
            'key': item.get('key'),
            'name': item.get('name'),
            'filename': item.get('filename'),
            'enabled': bool(item.get('enabled', True)),
            'strength': float(item.get('strength', 1.0)),
        })
    return JSONResponse({'ok': True, 'items': items})

@app.post("/api/lora/add")
async def add_lora_api(payload: dict = Body(...)):
    model = _get_model()
    if model is None:
        return JSONResponse({'ok': False, 'error': 'Model missing - Run Cell 2'}, status_code=400)

    raw_url = (payload.get('url') or '').strip()
    display_name = (payload.get('name') or '').strip()
    supplied_token = (payload.get('token') or '').strip()
    session_token = _get_civitai_token()
    effective_token = supplied_token or session_token
    if supplied_token:
        _set_civitai_token(supplied_token)
    if not raw_url:
        return JSONResponse({'ok': False, 'error': 'Missing CivitAI link.'}, status_code=400)

    url = _normalize_civitai_url(raw_url, effective_token)
    with gpu_lock:
        try:
            local_path, filename, final_url = _download_file(url, _custom_dir(), effective_token)
            key = f"{_slugify(display_name or os.path.splitext(filename)[0])}-{int(time.time()*1000)}"
            adapter_name = f"custom_{key}"
            model.load_lora_weights(local_path, adapter_name=adapter_name)
            item = {
                'key': key,
                'name': display_name or os.path.splitext(filename)[0],
                'filename': filename,
                'path': local_path,
                'source_url': final_url,
                'enabled': True,
                'strength': 1.0,
                'adapter_name': adapter_name,
                'loaded': True,
            }
            _custom_registry().append(item)
            _apply_active_loras(model)
            _save_registry()
            return JSONResponse({'ok': True, 'item': {
                'key': item['key'],
                'name': item['name'],
                'filename': item['filename'],
                'enabled': item['enabled'],
                'strength': item['strength'],
            }})
        except Exception as e:
            return JSONResponse({'ok': False, 'error': f'LoRA load failed: {e}. If this is a gated or private CivitAI file, save your CivitAI token in the UI first.'}, status_code=500)

@app.post("/api/lora/update")
async def update_lora_api(payload: dict = Body(...)):
    model = _get_model()
    if model is None:
        return JSONResponse({'ok': False, 'error': 'Model missing - Run Cell 2'}, status_code=400)

    key = payload.get('key')
    target = None
    for item in _custom_registry():
        if item.get('key') == key:
            target = item
            break
    if target is None:
        return JSONResponse({'ok': False, 'error': 'LoRA not found.'}, status_code=404)

    if 'enabled' in payload:
        target['enabled'] = bool(payload.get('enabled'))
    if 'strength' in payload and payload.get('strength') is not None:
        target['strength'] = max(0.0, min(2.0, float(payload.get('strength'))))

    with gpu_lock:
        try:
            _apply_active_loras(model)
            _save_registry()
            return JSONResponse({'ok': True, 'item': {
                'key': target['key'],
                'name': target['name'],
                'filename': target['filename'],
                'enabled': bool(target.get('enabled', True)),
                'strength': float(target.get('strength', 1.0)),
            }})
        except Exception as e:
            return JSONResponse({'ok': False, 'error': f'LoRA update failed: {e}'}, status_code=500)

# --- THE BACKEND (FASTAPI + GPU) ---
@app.get("/")
def home():
    return HTMLResponse(content=html_content)

@app.get("/api/generate")
async def generate_api(
    prompt: str,
    neg_prompt: str = "worst quality, low quality",
    grunge: str = "false",
    steps: int = 4,
    cfg: float = 1.5,
    width: int = 480,
    height: int = 720,
    seed: int = Query(-1)
):
    model = getattr(__main__, 'loaded_model', None)
    if not model:
        return {"error": "Model missing - Run Cell 2"}

    if grunge.lower() == "true":
        prompt = f'{prompt}, glitch art, grunge style, abstract horror, chromatic aberration, striking composition'

    full_prompt = f'masterpiece, best quality, newest, absurdres, {prompt}'

    with gpu_lock:
        used_seed = seed if seed is not None and int(seed) >= 0 else random.randint(0, 2147483647)
        generator = torch.Generator(device=DEVICE).manual_seed(int(used_seed))

        _apply_active_loras(model)
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.float16):
            result = model(
                prompt=full_prompt,
                negative_prompt=neg_prompt,
                num_inference_steps=steps,
                guidance_scale=cfg,
                width=width,
                height=height,
                generator=generator
            )

    img_byte_arr = io.BytesIO()
    result.images[0].save(img_byte_arr, format="JPEG", quality=90)
    img_byte_arr.seek(0)

    return StreamingResponse(
        img_byte_arr,
        media_type="image/jpeg",
        headers={
            "Cache-Control": "no-store, no-cache, must-revalidate, max-age=0",
            "X-Seed": str(used_seed)
        }
    )

# --- THE LAUNCHER (CLOUDFLARE SETUP) ---
PORT = 8000
print("🌐 [1/2] Establishing Secure Cloudflare Tunnel...")

with open("tunnel.log", "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stdout=f, stderr=f)

url = ""
for _ in range(30):
    time.sleep(0.5)
    try:
        with open("tunnel.log", "r") as f:
            log_content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
            if match:
                url = match.group(0)
                break
    except: pass

print("="*60)
if url:
    print(f"🔗 STUDIO LINK: {url}")
    print("="*60)
    print("If Cloudflare gives a security warning, simply click 'Visit Site'.")
else:
    print("⚠️ Cloudflare is taking too long. Check tunnel.log.")
print("="*60)
print("🛑 STARTING LOCAL SERVER... THE CELL IS NOW LOCKED AND FULLY ACTIVE.")

# --- NATIVE SERVER KEEP-ALIVE ---
config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="error")
server = uvicorn.Server(config)
await server.serve()

🌐 [1/2] Establishing Secure Cloudflare Tunnel...
🔗 STUDIO LINK: https://panels-scene-sentences-adaptation.trycloudflare.com
If Cloudflare gives a security warning, simply click 'Visit Site'.
🛑 STARTING LOCAL SERVER... THE CELL IS NOW LOCKED AND FULLY ACTIVE.


  0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]